In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataX") \
    .getOrCreate()

**What is Delta Lake?**

Key Concepts to Teach:
- Built on top of Parquet
- Supports ACID, schema enforcement, versioning
- Works best on Databricks and supports MERGE, UPDATE, DELETE

In [0]:
data = [("101","Alice","23-01-2026",1000),("102","Bob","23-01-2026",2000),(103,"Charlie","23-01-2026",1500)]
cols = ["id","cust_name","order_date","amount"]

df = spark.createDataFrame(data,cols)
display(df)

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("bronze_layer")

In [0]:
display(spark.sql("SELECT * FROM bronze_layer"))

**Layers in Architecture (Medallion):**
- Bronze ➝ Raw data
- Silver ➝ Cleaned + enriched data
- Gold ➝ Aggregated insights


Build Medallion Architecture (Practical)
🔶Bronze – Ingest Raw CSV

In [0]:
#bronze layer - Ingest from csv

df_raw = spark.read.option("header",True).csv("/databricks-datasets/retail-org/customers")
df_raw.write.format("delta").mode("overwrite").saveAsTable("customer_retail")
display(df_raw)

In [0]:
display(df_raw)

In [0]:
#silver Layer - Cleaned and standard data

df_bronze = spark.read.format("delta").load("tmp/customer_retail/_delta_log")

from pyspark.sql.functions import col, trim, lower

df_silver = df_bronze.withColumn("customer_name", lower(trim(col("customer_name")))).dropna(subset=["customer_name","customer_id"])

df_silver.write.format("delta").mode("overwrite").saveAsTable("customer_retail")
display(df_silver)
    

